In [10]:
# !conda install -c conda-forge healpy -y

In [18]:
pip install astropy

Note: you may need to restart the kernel to use updated packages.


In [19]:
pip install cfod

Note: you may need to restart the kernel to use updated packages.


In [20]:
pip install emcee corner

Note: you may need to restart the kernel to use updated packages.


In [24]:
# 1. Standard Python Libraries ---
import os                                                           # Handles file paths and directory checks
import sys                                                          # Access to system parameters and exit functions
import logging                                                      # Tracks events, errors, and status updates
import argparse                                                     # Manages input arguments (useful for script settings)
import pandas as pd               
# Data manipulation and analysis
# 2. Math & Statistics ---
os.environ["OMP_NUM_THREADS"]= "1"
os.environ["MKL_NUM_THREADS"]= "1"
os.environ["OPENBLAS_NUM_THREADS"]= "1"
os.environ["NUMEXPR_NUM_THREADS"]= "1"
os.environ["VECLIB_MAXIMUM_THREADS"]= "1"


# import numpy as np                                                # The core library for numerical operations and arrays
from scipy.stats import poisson, norm                               # specific probability distributions for statistical analysis

# 3. Astronomy & Cosmology ---
from astropy import units as u                                      # Handles physical units (Mpc, km/s, degrees, etc.)
from astropy.cosmology import Planck15                              # Pre-defined cosmology model (Planck 2015 parameters)
from astropy.cosmology import z_at_value                            # Helper to convert distance back to redshift
from astropy.table import Table                                     # Efficient handling of data tables (like DataFrames)
from astropy.io import ascii                                        # Reads/writes ASCII text files (common in astronomy)

# 4. MCMC & Fitting ---
import emcee                                                        # The MCMC sampler (Markov Chain Monte Carlo)

# 5. Visualization & Plotting ---
import matplotlib.pyplot as plt                                     # Standard plotting library
import corner                                                       # Specialized tool for making MCMC corner plots
from scipy.optimize import fsolve
from astropy.cosmology import Planck18                              # Pre-defined cosmology model (Planck 2018 parameters)
from astropy.coordinates import SkyCoord                            # Helper for astronomical coordinate transformations
import seaborn as sns                                               # Statistical data visualization based on Matplotlib
#import cfod                                                        # Importing the cfod module
import numpy as np                                                  # Numerical computations, array handling                                                  # Import the argparse module
import multiprocessing                                              # Import the multiprocessing module
from scipy.integrate import quad
import healpy as hp                                                 # Healpix library for spherical sky maps

### **1. All Declarations**

In [25]:
# step 0. Configuration and Global Constants ---
cosmo = Planck18 # Using Planck 2015 cosmology

# Define the custom unit 'bursts'
bursts = u.def_unit('bursts', represents=u.dimensionless_unscaled)
u.add_enabled_units([bursts])

# Normalization constants for host galaxy properties (Milky Way-like values)
MW_SFR = 1.65 * u.M_sun / u.yr
MW_STELLAR_MASS = 6.0e10 * u.M_sun

# Default bandwidth for observations (e.g., CHIME, assumes 400-800 MHz)
DEFAULT_OBSERVATION_BANDWIDTH = (800 - 400) * u.MHz # 400 MHz


# Step 1: Define day-scale quantities
R_central_day = 525.0
R_sigma_day = np.sqrt(30.0**2 + ((142.0 + 131.0)/2.0)**2)

# Step 2: Convert to yearly
R_central_year = R_central_day * 365.25
R_sigma_year = R_sigma_day * 365.25

# Step 3: Convert to astropy units
CHIME_OBSERVED_RATE_REPORTED = (R_central_year * bursts) / u.yr
CHIME_OBSERVED_RATE_TOTAL_ERROR = (R_sigma_year * bursts) / u.yr

# Step 4: Convert to log10 space (AFTER defining above)
LOG10_R_MEAN = np.log10(R_central_year)
LOG10_R_SIGMA = R_sigma_year / (R_central_year * np.log(10))  # error propagation

# Step 5: Prior bounds
log10_R_min = LOG10_R_MEAN - 5 * LOG10_R_SIGMA
log10_R_max = LOG10_R_MEAN + 5 * LOG10_R_SIGMA 

PRIOR_BOUNDS = {
    'log10_R_all_sky': (log10_R_min, log10_R_max),
    'log10_E_min_pop': (30.0, 38.0),
    'gamma': (0.0, 3.5),
    'log10_E_star': (40.0, 43.0),
    'alpha': (-3.0, 1.0)
}

In [27]:
# Testing the code
print("1. Chime Observed Rate burst per year :" , CHIME_OBSERVED_RATE_REPORTED)
print("2. R mean testing :", LOG10_R_MEAN, LOG10_R_SIGMA)
print("3. Prior Bound :", PRIOR_BOUNDS)

1. Chime Observed Rate burst per year : 191756.25 bursts / yr
2. R mean testing : 5.282749528012292 0.11561152904015505
3. Prior Bound : {'log10_R_all_sky': (np.float64(4.704691882811517), np.float64(5.860807173213066)), 'log10_E_min_pop': (30.0, 38.0), 'gamma': (0.0, 3.5), 'log10_E_star': (40.0, 43.0), 'alpha': (-3.0, 1.0)}


### **2. Data Extraction and filtering process**

### **3. Cleaned Data File Loading Function definition**

In [28]:
def load_galaxy_catalog(file_path: str) -> Table:                             # Define function that takes file path (string) and returns an Astropy Table
    """
    Load and standardize a galaxy catalog for Bayesian analysis.

    This function reads a galaxy catalog file from the specified path using
    Astropy's ASCII reader and transforms the dataset into a clean, consistent
    format suitable for downstream statistical modeling (e.g., Bayesian inference,
    likelihood evaluation, or population studies).

    The function performs the following key operations:
    1. Safely loads the input catalog file with error handling.
    2. Verifies the presence of all required columns in the dataset.
    3. Renames columns from their original (often verbose or inconsistent)
       names into standardized, Python-friendly identifiers.
    4. Constructs a new Astropy Table containing only the required and renamed
       columns for further analysis.
    5. Logs detailed information about the loading process and column mappings
       for transparency and debugging.

    Parameters
    ----------
    file_path : str
        Path to the input galaxy catalog file.
        The file must be readable by `astropy.io.ascii.read` and contain
        all required columns listed in `column_mappings`.

    Returns
    -------
    Table
        An `astropy.table.Table` object containing the processed galaxy catalog.
        The table includes standardized column names such as:
        - galaxy_name : Identifier of the galaxy
        - ra_deg, dec_deg : Sky coordinates in degrees
        - d25_semi_major_arcmin, d25_semi_minor_arcmin : Galaxy size parameters
        - d25_pos_angle_deg : Orientation angle
        - inclination_angle_deg : Galaxy inclination
        - luminosity_distance_mpc : Distance to galaxy
        - log_sfr_hec : Logarithmic star formation rate
        - log10_stellar_mass_hec : Logarithmic stellar mass
        - total_observation_time_hr : Observation exposure time

    Raises
    ------
    FileNotFoundError
        If the specified file path does not exist.

    KeyError
        If any required column is missing from the input catalog.

    Exception
        For any other unexpected errors encountered during file reading.

    Notes
    -----
    - This function ensures consistency in column naming, which is critical
      for reproducible scientific analysis and integration with Bayesian models.
    - The output table is intentionally restricted to only the required fields
      to avoid ambiguity and reduce downstream processing complexity.
    - Logging is used extensively to aid debugging and provide traceability
      in large data-processing pipelines.

    """

    logging.info(f"Attempting to load galaxy catalog from '{file_path}'...")  # Log message to indicate loading has started

    try:                                                                      # Begin error-handling block to safely read file
        catalog = ascii.read(file_path)                                       # Read input file into Astropy Table (structured data format)

    except FileNotFoundError:                                                 # If file path is incorrect or file does not exist
        logging.error(f"Error: The file '{file_path}' was not found.")        # Log error message
        raise                                                                 # Re-raise error so main program stops execution properly

    except Exception as e:                                                    # Catch any other unexpected errors during file reading
        logging.error(f"Error reading file '{file_path}': {e}")               # Log detailed error message
        raise                                                                 # Re-raise exception for outer handling

    column_mappings = {                                                       # Dictionary to rename original column names to standardized names
        'Galaxy name': 'galaxy_name',                                         # Rename galaxy name column to clean Python-friendly format
        'RAJ2000 (deg)': 'ra_deg',                                            # Right Ascension in degrees (shortened for convenience)
        'DEJ2000 (deg)': 'dec_deg',                                           # Declination in degrees (used for sky position filtering)
        'D25 semi-major axis (arcmin)': 'd25_semi_major_arcmin',              # Galaxy major axis size
        'D25 semi-minor axis (arcmin)': 'd25_semi_minor_arcmin',              # Galaxy minor axis size
        'D25 positional angle, North-to-Northeast (deg)': 'd25_pos_angle_deg', # Orientation angle of galaxy ellipse
        'Inclination angle (deg)': 'inclination_angle_deg',                    # Inclination angle (important for selection cuts)
        'Luminosity distance (Mpc)': 'luminosity_distance_mpc',                # Distance to galaxy (used in astrophysical calculations)
        'logSFR_HEC': 'log_sfr_hec',                                           # Log star formation rate
        'logM_HEC': 'log10_stellar_mass_hec',                                  # Log stellar mass
        'mean_exposure_Hr': 'total_observation_time_hr',                       # Observation time (affects detection probability)
    }

    new_catalog_data = {}                                                      # Create empty dictionary to store cleaned/renamed data

    for original_col, new_col in column_mappings.items():                      # Loop over each mapping pair
        if original_col in catalog.colnames:                                   # Check if required column exists in input data
            new_catalog_data[new_col] = catalog[original_col]                  # Copy data into new dictionary with renamed column
        else:                                                                  # If required column is missing
            logging.error(                                                     # Log error message with details
                f"Required column '{original_col}' not found in the input catalog. "
                f"Available columns are: {catalog.colnames}"
            )
            raise KeyError(f"Missing column: {original_col}")                  # Raise error to stop execution (critical data missing)

    processed_catalog = Table(new_catalog_data)                                # Convert cleaned dictionary into Astropy Table

    logging.info(f"Successfully loaded catalog with {len(processed_catalog)} galaxies.")  # Log number of galaxies loaded
    logging.info("Renamed columns:")                                           # Log header for column mapping output
    for original, new in column_mappings.items():                              # Loop through mappings again for display
        logging.info(f"  '{original}' -> '{new}'")                             # Log each renaming for transparency/debugging

    return processed_catalog                                                   # Return final cleaned and standardized catalog for further analysis

In [29]:
# --- Test script for load_galaxy_catalog ---                          # Section header indicating purpose of this block

# File path here                                                       # Define input file location
file_path = "LVG_table.txt"  # Path to galaxy catalog file

try:                                                                  # Begin try block to safely execute function
    table = load_galaxy_catalog(file_path)                            # Call function to load and process catalog

    print("\n FUNCTION EXECUTED SUCCESSFULLY!\n")                     # Confirmation message if no error occurs

    # Show first 5 rows                                               # Display preview of dataset
    print(" First 5 rows of table:")                                  # Label for output section
    print(table[:5])                                                  # Print first five entries of the table

    # Show column names                                               # Display renamed column structure
    print("\n Column names after renaming:")                          # Label for column names output
    print(table.colnames)                                             # Print list of column names

    # Show number of galaxies                                         # Display dataset size
    print(f"\n Total galaxies loaded: {len(table)}")                  # Print total number of rows (galaxies)

except Exception as e:                                                # Catch any error during execution
    print("\nERROR OCCURRED:")                                        # Error message header
    print(e)                                                          # Print actual error details


 FUNCTION EXECUTED SUCCESSFULLY!

 First 5 rows of table:
galaxy_name  ra_deg  ... log10_stellar_mass_hec total_observation_time_hr
----------- -------- ... ---------------------- -------------------------
   UGC00092 2.508291 ...                 9.2305                     17.61
   UGC00132 3.503208 ...                 8.7558                      5.29
    NGC0100 6.012109 ...                 9.5165                     18.29
   UGC00313 7.858833 ...                 9.0369                     16.38
    NGC0145 7.940624 ...                 9.4395                     17.62

 Column names after renaming:
['galaxy_name', 'ra_deg', 'dec_deg', 'd25_semi_major_arcmin', 'd25_semi_minor_arcmin', 'd25_pos_angle_deg', 'inclination_angle_deg', 'luminosity_distance_mpc', 'log_sfr_hec', 'log10_stellar_mass_hec', 'total_observation_time_hr']

 Total galaxies loaded: 814


In [31]:
# --- 0. Configuration and Global Constants ---
cosmo = Planck15                                                            # Using Planck 2015 cosmology

# Define the custom unit 'bursts'
bursts = u.def_unit('bursts', represents=u.dimensionless_unscaled)
u.add_enabled_units([bursts])

# Normalization constants for host galaxy properties (Milky Way-like values)
MW_SFR = 1.65 * u.M_sun / u.yr
MW_STELLAR_MASS = 6.0e10 * u.M_sun

# Default bandwidth for observations (e.g., CHIME, assumes 400-800 MHz)
DEFAULT_OBSERVATION_BANDWIDTH = (800 - 400) * u.MHz # 400 MHz

# --- CHIME All-Sky Rate (Correct Order) ---

# Step 1: Define day-scale quantities
R_central_day = 525.0
R_sigma_day = np.sqrt(30.0**2 + ((142.0 + 131.0)/2.0)**2)

# Step 2: Convert to yearly
R_central_year = R_central_day * 365.25
R_sigma_year = R_sigma_day * 365.25

# Step 3: Convert to astropy units
CHIME_OBSERVED_RATE_REPORTED = (R_central_year * bursts) / u.yr
CHIME_OBSERVED_RATE_TOTAL_ERROR = (R_sigma_year * bursts) / u.yr

# Step 4: Convert to log10 space (AFTER defining above)
LOG10_R_MEAN = np.log10(R_central_year)
LOG10_R_SIGMA = R_sigma_year / (R_central_year * np.log(10))  # error propagation

# Step 5: Prior bounds
log10_R_min = LOG10_R_MEAN - 5 * LOG10_R_SIGMA
log10_R_max = LOG10_R_MEAN + 5 * LOG10_R_SIGMA

PRIOR_BOUNDS = {
    'log10_R_all_sky': (log10_R_min, log10_R_max),
    'log10_E_min_pop': (30.0, 38.0),
    'gamma': (0.0, 3.0),
    'log10_E_star': (40.0, 43.0),
    'alpha': (-3.0, 1.0)
}







# --- 1. Data Loading Function ---
def load_galaxy_catalog(file_path: str) -> Table:
    """
    Loads the galaxy catalog from a specified file path, renaming columns
    to a consistent format for the Bayesian framework.
    """
    logging.info(f"Attempting to load galaxy catalog from '{file_path}'...")
    try:
        catalog = ascii.read(file_path)
    except FileNotFoundError:
        logging.error(f"Error: The file '{file_path}' was not found.")
        raise # Re-raise to be caught by main block for exit
    except Exception as e:
        logging.error(f"Error reading file '{file_path}': {e}")
        raise # Re-raise to be caught by main block for exit

    column_mappings = {
        'Galaxy name': 'galaxy_name',
        'RAJ2000 (deg)': 'ra_deg',
        'DEJ2000 (deg)': 'dec_deg',
        'D25 semi-major axis (arcmin)': 'd25_semi_major_arcmin',
        'D25 semi-minor axis (arcmin)': 'd25_semi_minor_arcmin',
        'D25 positional angle, North-to-Northeast (deg)': 'd25_pos_angle_deg',
        'Inclination angle (deg)': 'inclination_angle_deg',
        'Luminosity distance (Mpc)': 'luminosity_distance_mpc',
        'logSFR_HEC': 'log_sfr_hec',
        'logM_HEC': 'log10_stellar_mass_hec',
        'mean_exposure_Hr': 'total_observation_time_hr',
    }

    new_catalog_data = {}
    for original_col, new_col in column_mappings.items():
        if original_col in catalog.colnames:
            new_catalog_data[new_col] = catalog[original_col]
        else:
            logging.error(
                f"Required column '{original_col}' not found in the input catalog. "
                f"Available columns are: {catalog.colnames}"
            )
            raise KeyError(f"Missing column: {original_col}") # Re-raise for outer handling

    processed_catalog = Table(new_catalog_data)

    logging.info(f"Successfully loaded catalog with {len(processed_catalog)} galaxies.")
    logging.info("Renamed columns:")
    for original, new in column_mappings.items():
        logging.info(f"  '{original}' -> '{new}'")

    return processed_catalog








# --- 2. Core Helper Functions ---

def _schechter_function_integrand(E_norm, gamma):
    """
    The integrand for the Schechter-like energy distribution.
    E_norm: Dimensionless energy (E / E_star)
    gamma: Power-law index (dimensionless).
    """
    E_norm = np.maximum(E_norm, 1e-100) # Small positive floor for numerical stability
    return E_norm**(-gamma) * np.exp(-E_norm)

def _integrate_schechter(E_min_iso: u.Quantity, E_star: u.Quantity, gamma: float, num_integration_points: int = 10000) -> u.Quantity:
    """
    Numerically integrates the Schechter function from E_min_iso to infinity using np.trapezoid.
    Returns: astropy.units.Quantity (dimensionless_unscaled) or -np.inf if integral fails.
    """
    if E_star.to_value(u.erg) <= 0:
        return 0.0 * u.dimensionless_unscaled

    E_min_norm = (E_min_iso / E_star).to_value(u.dimensionless_unscaled)

    if E_min_norm <= 0: # Should not happen with valid energies
        return -np.inf * u.dimensionless_unscaled

    x_max_norm = max(E_min_norm * 10, 1000.0)

    if x_max_norm <= E_min_norm:
        x_max_norm = E_min_norm + 1.0

    X_values = np.logspace(np.log10(E_min_norm), np.log10(x_max_norm), num=num_integration_points)

    Y_values = _schechter_function_integrand(X_values, gamma)

    if not np.all(np.isfinite(Y_values)):
        return -np.inf * u.dimensionless_unscaled

    integral_result = np.trapezoid(Y_values, X_values)

    if not np.isfinite(integral_result) or integral_result < 0:
        return -np.inf * u.dimensionless_unscaled

    return integral_result * u.dimensionless_unscaled


def sfrd(z: float) -> u.Quantity:
    """
    Cosmic Star Formation Rate Density (SFRD) as a function of redshift.
    Returns: astropy.units.Quantity in M_sun / yr / Mpc^3.
    """
    sfrd_val = 0.015 * (1 + z)**2.7 / (1 + ((1 + z) / 2.9)**5.6) * (u.M_sun / u.yr / u.Mpc**3)
    return sfrd_val.to(u.M_sun / u.yr / u.Mpc**3)


# def total_stellar_mass_density(z: float) -> u.Quantity:
#     """
#     Cosmic Stellar Mass Density (CSMD) as a function of redshift.
#     Kept for flexibility if model_property_type is changed to 'stellar_mass'.
#     Returns: astropy.units.Quantity in M_sun / Mpc^3.
#     """
#     csm_val = 7e8 * (1 + z)**(-0.5) * np.exp(-z/5) * (u.M_sun / u.Mpc**3)
#     return csm_val.to(u.M_sun / u.Mpc**3)


def total_stellar_mass_density(z, R=0.27):
    integrand = lambda z_prime: sfrd(z_prime).value / (
        (1 + z_prime) * cosmo.H(z_prime).to(u.yr**-1).value
    )

    integral_value, _ = quad(
        integrand, z, np.inf,
        limit=1000, epsrel=1e-8, epsabs=1e-10
    )

    csmd = (1 - R) * integral_value

    return (csmd / 6.08e10) / (1 + z) * (u.M_sun / u.Mpc**3)



def get_differential_comoving_volume(z: float) -> u.Quantity:
    """
    Calculates the differential comoving volume dVc/dz/dOmega at redshift z.
    Returns: astropy.units.Quantity in Mpc^3 / sr.
    """
    return cosmo.differential_comoving_volume(z).to(u.Mpc**3 / u.sr)


def _convert_fluence_to_isotropic_energy_with_bandwidth(
        fluence_Jy_ms: u.Quantity,
        z: float,
        observation_bandwidth: u.Quantity,
        alpha: float) -> u.Quantity:

    """
    Converts observed fluence (Jy ms) to isotropic equivalent energy (erg)
    considering observation bandwidth and spectral index alpha.
    """

    fluence_erg_cm2_Hz = fluence_Jy_ms.to(u.erg / (u.cm**2 * u.Hz))

    D_L = cosmo.luminosity_distance(z).to(u.cm)

    total_fluence_erg_cm2 = (fluence_erg_cm2_Hz * observation_bandwidth).to(u.erg / u.cm**2)

    redshift_term = (1 + z)**(2 + alpha)

    if not np.isfinite(redshift_term) or redshift_term <= 0:
        return np.inf * u.erg

    E_iso = (4 * np.pi * D_L**2 * total_fluence_erg_cm2 / redshift_term).to(u.erg)

    return E_iso
    

def _get_effective_minimum_energy(
        z: float,
        fluence_threshold_Jy_ms: u.Quantity,
        observation_bandwidth: u.Quantity,
        E_min_population: u.Quantity,
        alpha: float) -> u.Quantity:

    """
    Calculates the effective minimum detectable isotropic energy.
    """

    E_min_from_fluence = _convert_fluence_to_isotropic_energy_with_bandwidth(
        fluence_threshold_Jy_ms,
        z,
        observation_bandwidth,
        alpha
    )

    if not np.isfinite(E_min_from_fluence.value):
        return np.inf * u.erg

    effective_E_min = max(
        E_min_population.to_value(u.erg),
        E_min_from_fluence.to_value(u.erg)
    ) * u.erg

    return effective_E_min







# --- 3. Main Rate Model Functions ---

def all_sky_rate_model(params: tuple,
                       global_fluence_threshold_Jy_ms: u.Quantity,
                       model_property_type: str = 'sfr',
                       z_max_integration: float = 2.5,
                       num_redshift_points: int = 100,
                       global_observation_bandwidth: u.Quantity = DEFAULT_OBSERVATION_BANDWIDTH) -> u.Quantity:

    """
    Calculates the theoretical all-sky FRB rate above a fluence threshold.
    """

    log10_R_all_sky, log10_E_min_pop, gamma, log10_E_star, alpha = params

    R_all_sky = (10**log10_R_all_sky) * bursts / u.yr
    E_min_pop = (10**log10_E_min_pop) * u.erg
    E_star = (10**log10_E_star) * u.erg

    redshifts = np.logspace(np.log10(1e-4), np.log10(z_max_integration), num_redshift_points)

    integrand_values = []

    for z in redshifts:

        if model_property_type == 'sfr':
            cosmic_density = sfrd(z)
            normalization_factor = MW_SFR
        else:
            cosmic_density = total_stellar_mass_density(z)
            normalization_factor = MW_STELLAR_MASS

        density_ratio_per_volume = (cosmic_density / normalization_factor).to(1/u.Mpc**3)

        E_th = _get_effective_minimum_energy(
            z,
            global_fluence_threshold_Jy_ms,
            global_observation_bandwidth,
            E_min_pop,
            alpha
        )

        numerator = _integrate_schechter(E_th, E_star, gamma)
        denominator = _integrate_schechter(E_min_pop, E_star, gamma)

        if denominator.value <= 0:
            return -np.inf * bursts/u.yr

        energy_fraction = numerator / denominator

        dVc_dz_dOmega = get_differential_comoving_volume(z)

        # term = (
        #     density_ratio_per_volume *
        #     energy_fraction *
        #     dVc_dz_dOmega *
        #     (4*np.pi*u.sr)
        # ).to_value()
        term = (
            density_ratio_per_volume *
            energy_fraction *
            dVc_dz_dOmega *
            (4 * np.pi * u.sr) /
            (1 + z)   #  cosmic time dilation correction
        )

        integrand_values.append(term)

    integral = np.trapezoid(integrand_values, redshifts)

    predicted_rate = R_all_sky * integral

    return predicted_rate.to(bursts/u.yr)





def galaxy_rate_model(params: tuple,
                      z: float,
                      galaxy_property_value: u.Quantity,
                      fluence_threshold_Jy_ms: u.Quantity,
                      model_property_type: str = 'sfr',
                      observation_bandwidth: u.Quantity = DEFAULT_OBSERVATION_BANDWIDTH) -> u.Quantity:

    """
    Calculates the FRB rate for a single galaxy.
    """

    log10_R_all_sky, log10_E_min_pop, gamma, log10_E_star, alpha = params

    R_all_sky = (10**log10_R_all_sky) * bursts / u.yr
    E_min_pop = (10**log10_E_min_pop) * u.erg
    E_star = (10**log10_E_star) * u.erg

    if model_property_type == 'sfr':
        normalization_factor = MW_SFR
    else:
        normalization_factor = MW_STELLAR_MASS

    property_ratio = (galaxy_property_value / normalization_factor).to_value()

    E_th = _get_effective_minimum_energy(
        z,
        fluence_threshold_Jy_ms,
        observation_bandwidth,
        E_min_pop,
        alpha
    )

    numerator = _integrate_schechter(E_th, E_star, gamma)
    denominator = _integrate_schechter(E_min_pop, E_star, gamma)

    if denominator.value <= 0:
        return -np.inf * bursts/u.yr

    predicted_rate = R_all_sky * property_ratio * (numerator / denominator)

    return predicted_rate.to(bursts/u.yr)










# --- 4. Bayesian Framework Functions (Prior, Likelihood, Posterior) ---

def log_prior(params: tuple):

    log10_R_all_sky, log10_E_min_pop, gamma, log10_E_star, alpha = params

    if not (PRIOR_BOUNDS['log10_R_all_sky'][0] <= log10_R_all_sky <= PRIOR_BOUNDS['log10_R_all_sky'][1]):
        return -np.inf

    log_prior_R = norm.logpdf(
        log10_R_all_sky,
        loc=LOG10_R_MEAN,
        scale=LOG10_R_SIGMA
    )

    # if not (PRIOR_BOUNDS['gamma'][0] <= gamma <= PRIOR_BOUNDS['gamma'][1]):
    #     return -np.inf

    # log_prior_gamma = norm.logpdf(
    #     gamma,
    #     loc=1.4,
    #     scale=0.9
    # )
    # Check gamma within bounds
    if not (PRIOR_BOUNDS['gamma'][0] <= gamma <= PRIOR_BOUNDS['gamma'][1]):
        return -np.inf
    if gamma <= 0:
        return -np.inf

    # Gaussian prior on gamma
    log_prior_gamma = norm.logpdf(
        gamma,
        loc=1.4,
        scale=0.9
    )

    if not (PRIOR_BOUNDS['alpha'][0] <= alpha <= PRIOR_BOUNDS['alpha'][1]):
        return -np.inf

    log_prior_alpha = norm.logpdf(alpha, loc=-1.5, scale=1.0)

    if not (PRIOR_BOUNDS['log10_E_min_pop'][0] <= log10_E_min_pop <= PRIOR_BOUNDS['log10_E_min_pop'][1]):
        return -np.inf

    if not (PRIOR_BOUNDS['log10_E_star'][0] <= log10_E_star <= PRIOR_BOUNDS['log10_E_star'][1]):
        return -np.inf

    if log10_E_star <= log10_E_min_pop:
        return -np.inf

    log_prior_E_star = norm.logpdf(
        log10_E_star,
        loc=41.38,
        scale=0.51
    )
    return log_prior_R + log_prior_gamma + log_prior_alpha + log_prior_energy










def log_likelihood(params: tuple, observed_all_sky_data: dict, obs_galaxy_data_list: list, model_property_type: str) -> float:
    """
    Calculates the total log-likelihood of the model given the data.
    """
    total_log_likelihood = 0.0

    chime_obs_rate = observed_all_sky_data['rate']
    chime_obs_error = observed_all_sky_data['error']
    chime_fluence_thresh = observed_all_sky_data['fluence_threshold']
    chime_bandwidth = observed_all_sky_data['bandwidth']

    try:
        predicted_all_sky_rate_per_year = all_sky_rate_model(
            params, chime_fluence_thresh, model_property_type, global_observation_bandwidth=chime_bandwidth
        )

        if not np.isfinite(predicted_all_sky_rate_per_year.value):
            return -np.inf

        predicted_all_sky_rate_chime_units = (predicted_all_sky_rate_per_year / (4 * np.pi * u.steradian)).to(bursts / u.day / u.steradian).to_value(chime_obs_rate.unit)

        log_likelihood_all_sky = norm.logpdf(predicted_all_sky_rate_chime_units,
                                             loc=chime_obs_rate.to_value(chime_obs_rate.unit),
                                             scale=chime_obs_error.to_value(chime_obs_error.unit))
        total_log_likelihood += log_likelihood_all_sky

    except Exception as e:
        # In a multiprocessing context, logging from child processes can be tricky.
        # It's often better to let the exception propagate and be caught by the pool.
        # For debugging, you might print, but for production, this might be too noisy.
        # logging.error(f"Error in all-sky rate likelihood calculation: {e}")
        return -np.inf


    galaxy_fluence_threshold = 5 * u.Jy * u.ms

    for i, gal_data in enumerate(obs_galaxy_data_list):
        try:
            gal_distance = gal_data['luminosity_distance_mpc']
            gal_obs_time = gal_data['total_observation_time_hr']
            gal_bandwidth = DEFAULT_OBSERVATION_BANDWIDTH

            redshift_result = z_at_value(cosmo.luminosity_distance, gal_distance)

            if isinstance(redshift_result, u.Quantity):
                gal_z = redshift_result.to_value(u.dimensionless_unscaled)
            elif np.isfinite(redshift_result):
                gal_z = float(redshift_result)
            else:
                # logging.error(f"Non-finite or invalid redshift for galaxy {gal_data.get('galaxy_name', i)}. Lum_dist: {gal_distance}")
                return -np.inf

            if model_property_type == 'sfr':
                gal_property_value = (10**gal_data['log_sfr_hec']) * (u.M_sun / u.yr)
            elif model_property_type == 'stellar_mass':
                gal_property_value = (10**gal_data['log10_stellar_mass_hec']) * u.M_sun
            else:
                # logging.error(f"Invalid model_property_type: {model_property_type}")
                return -np.inf

            R_model_gal = galaxy_rate_model(
                params, z=gal_z, galaxy_property_value=gal_property_value,
                fluence_threshold_Jy_ms=galaxy_fluence_threshold,
                model_property_type=model_property_type,
                observation_bandwidth=gal_bandwidth
            )

            if not np.isfinite(R_model_gal.value):
                # logging.error(f"Non-finite galaxy rate for galaxy {gal_data.get('galaxy_name', i)}")
                return -np.inf

            mu_gal = (R_model_gal * gal_obs_time).to_value(u.dimensionless_unscaled)

            if mu_gal < 0 or not np.isfinite(mu_gal):
                # logging.error(f"Invalid mu_gal ({mu_gal}) for galaxy {gal_data.get('galaxy_name', i)}")
                return -np.inf

            log_likelihood_galaxies_term = -mu_gal # Poisson likelihood: exp(-mu) for zero observed events
            total_log_likelihood += log_likelihood_galaxies_term

        except Exception as e:
            # logging.error(f"Error in galaxy likelihood calculation for galaxy {gal_data.get('galaxy_name', i)}: {e}")
            return -np.inf

    if not np.isfinite(total_log_likelihood):
        # logging.error("Total log likelihood is not finite.")
        return -np.inf

    return total_log_likelihood


def log_posterior(params: tuple, observed_all_sky_data: dict, obs_galaxy_data_list: list, model_property_type: str) -> float:
    """
    Calculates the log-posterior probability.
    """
    lp = log_prior(params)
    if not np.isfinite(lp):
        return -np.inf

    ll = log_likelihood(params, observed_all_sky_data, obs_galaxy_data_list, model_property_type)
    if not np.isfinite(ll):
        return -np.inf

    return lp + ll

def setup_logging(log_file_path: str):
    """
    Configures the logging system to output to both console and a file.
    """
    # Create directory for log file if it doesn't exist
    log_dir = os.path.dirname(log_file_path)
    if log_dir and not os.path.exists(log_dir):
        os.makedirs(log_dir, exist_ok=True)

    # Get the root logger
    logger = logging.getLogger()
    logger.setLevel(logging.INFO) # Set the minimum level of messages to handle

    # Clear any existing handlers to prevent duplicate output if called multiple times
    if logger.hasHandlers():
        logger.handlers.clear()

    # Create file handler
    file_handler = logging.FileHandler(log_file_path)
    # Define a format for log messages in the file (e.g., with timestamp and level)
    file_formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(file_formatter)
    logger.addHandler(file_handler)

    # Create console handler
    console_handler = logging.StreamHandler(sys.stdout)
    # Define a simpler format for console output
    console_formatter = logging.Formatter('%(levelname)s: %(message)s')
    console_handler.setFormatter(console_formatter)
    logger.addHandler(console_handler)

    logging.info(f"Logging configured. Output will be saved to '{log_file_path}'")


def parse_arguments():
    print("Entered")

    parser = argparse.ArgumentParser(
        description="Run MCMC for FRB rate modeling."
    )

    parser.add_argument("--catalog-path", type=str, required=True)
    parser.add_argument("--output-dir", type=str, required=True)
    parser.add_argument("--log-file", type=str, default="mcmc_run.log")
    parser.add_argument("--nwalkers", type=int, default=200)
    parser.add_argument("--nsteps", type=int, default=2000)
    parser.add_argument("--nburnin", type=int, default=500)
    parser.add_argument(
        "--model-property-type",
        type=str,
        choices=["sfr", "stellar_mass"],
        default="sfr"
    )

    # ADD THIS BACK
    parser.add_argument("--nthreads", type=int, default=1)

    # FORCE arguments in Colab
    forced_args = [
        "--catalog-path", "/kaggle/input/datasets/gaurav7615244/dataset/LVG_table.txt",
        "--output-dir", "/kaggle/working/MCMC1_RESULT",
        "--nthreads", "1"   # optional but clean
    ]

    args = parser.parse_args(forced_args)

    print("Parsed args:", args)
    return args











# --- 5. Main Execution Block for MCMC ---
if __name__ == '__main__':
    args = parse_arguments()

    # Create the output directory if it does not exist
    os.makedirs(args.output_dir, exist_ok=True)

    # --- Logging Setup ---
    log_file_full_path = os.path.join(args.output_dir, args.log_file)
    setup_logging(log_file_full_path)

    logging.info("--- Starting Bayesian FRB Rate Modeling with emcee ---")
    logging.info(f"Command-line arguments: {args}")

    # --- Data Setup ---
    observed_all_sky_data = {
        'rate': CHIME_OBSERVED_RATE_REPORTED,
        'error': CHIME_OBSERVED_RATE_TOTAL_ERROR,
        'fluence_threshold': 5 * u.Jy * u.ms,
        'bandwidth': DEFAULT_OBSERVATION_BANDWIDTH
    }
    logging.info(f"CHIME Observed Rate (for likelihood): {observed_all_sky_data['rate']:.4g}")

    # --- Galaxy Catalog Path ---
    galaxy_catalog_file_path = args.catalog_path
    if not os.path.exists(galaxy_catalog_file_path):
        logging.critical(f"Error: Galaxy catalog '{galaxy_catalog_file_path}' not found. "
                         "Please ensure the file path is correct. Exiting.")
        sys.exit(1) # Exit if the critical file is missing

    # Load galaxy data
    try:
        obs_galaxy_table = load_galaxy_catalog(galaxy_catalog_file_path)
    except Exception as e:
        logging.critical(f"Failed to load galaxy catalog. Exiting. Error: {e}")
        sys.exit(1)


    obs_galaxy_data_for_likelihood = []
    for row in obs_galaxy_table:
        obs_galaxy_data_for_likelihood.append({
            'luminosity_distance_mpc': row['luminosity_distance_mpc'] * u.Mpc,
            'log_sfr_hec': row['log_sfr_hec'],
            'log10_stellar_mass_hec': row['log10_stellar_mass_hec'],
            'total_observation_time_hr': row['total_observation_time_hr'] * u.hr,
        })
    logging.info(f"Prepared {len(obs_galaxy_data_for_likelihood)} galaxies for likelihood calculation.")

    # --- MCMC Setup ---
    MODEL_PROPERTY_TYPE = args.model_property_type
    ndim = 5
    nwalkers = args.nwalkers
    n_burn_in = args.nburnin
    n_steps = args.nsteps
    nthreads = args.nthreads # Number of threads from command-line arguments

    # Initialize walkers by sampling UNIFORMLY across the prior bounds
    p0 = np.zeros((nwalkers, ndim))
    param_keys = list(PRIOR_BOUNDS.keys())

    logging.info("\nInitializing walkers with robust checks (sampling uniformly across priors)...")
    for i in range(nwalkers):
        attempt = 0
        while True:
            candidate_p = np.array([
                np.random.uniform(PRIOR_BOUNDS[param_keys[0]][0], PRIOR_BOUNDS[param_keys[0]][1]), # log10_K0
                np.random.uniform(PRIOR_BOUNDS[param_keys[1]][0], PRIOR_BOUNDS[param_keys[1]][1]), # log10_E_min_pop
                np.random.uniform(PRIOR_BOUNDS[param_keys[2]][0], PRIOR_BOUNDS[param_keys[2]][1]), # gamma
                np.random.uniform(PRIOR_BOUNDS[param_keys[3]][0], PRIOR_BOUNDS[param_keys[3]][1]), # log10_E_star
                np.random.uniform(PRIOR_BOUNDS[param_keys[4]][0], PRIOR_BOUNDS[param_keys[4]][1])  # A (new parameter)
            ])

            # Explicitly check E_star > E_min_pop constraint
            if candidate_p[3] > candidate_p[1]:
            # if candidate_p[3] > candidate_p[1] and candidate_p[2] > 0:
                try:
                    # For initialization, we can directly call log_posterior without a pool
                    test_lp = log_posterior(tuple(candidate_p), observed_all_sky_data, obs_galaxy_data_for_likelihood, MODEL_PROPERTY_TYPE)
                    if np.isfinite(test_lp):
                        p0[i] = candidate_p
                        break
                except Exception as e:
                    pass # Continue to next attempt if exception occurs

            attempt += 1
            if attempt > 5000: # Increased attempts for robust sampling in higher dim
                logging.critical(f"Could not find a valid initial point for walker {i} after {attempt} attempts. "
                                 "This suggests your prior bounds or model constraints are too restrictive, "
                                 "or there's no valid parameter space. Check model and data carefully. Exiting.")
                sys.exit(1) # Exit if cannot initialize walkers
    logging.info(f"\nSuccessfully initialized {nwalkers} walkers with valid starting points.")

    logging.info(f"\nInitializing emcee sampler with {nwalkers} walkers and {ndim} dimensions.")
    logging.info(f"Model property type: '{MODEL_PROPERTY_TYPE}'")
    logging.info(f"Burn-in steps: {n_burn_in}, Production steps: {n_steps}")
    logging.info(f"Number of threads for multiprocessing: {nthreads}")

    # Use multiprocessing pool if nthreads > 1
    if nthreads > 1:
        with multiprocessing.Pool(nthreads) as pool:
            sampler = emcee.EnsembleSampler(
                nwalkers, ndim, log_posterior,
                args=(observed_all_sky_data, obs_galaxy_data_for_likelihood, MODEL_PROPERTY_TYPE),
                pool=pool
            )

            logging.info(f"Running {n_burn_in} burn-in steps...")
            state = sampler.run_mcmc(p0, n_burn_in, progress=True)
            sampler.reset()

            logging.info(f"Running {n_steps} production steps...")
            sampler.run_mcmc(state, n_steps, progress=True)
    else: # Run without multiprocessing
        sampler = emcee.EnsembleSampler(
            nwalkers, ndim, log_posterior,
            args=(observed_all_sky_data, obs_galaxy_data_for_likelihood, MODEL_PROPERTY_TYPE)
        )

        logging.info(f"Running {n_burn_in} burn-in steps...")
        state = sampler.run_mcmc(p0, n_burn_in, progress=True)
        sampler.reset()

        logging.info(f"Running {n_steps} production steps...")
        sampler.run_mcmc(state, n_steps, progress=True)


    full_chain = sampler.get_chain()
    full_log_probs = sampler.get_log_prob()

    # Save outputs to the specified output directory
    full_chain_filename = os.path.join(args.output_dir, f"mcmc_full_chain_{MODEL_PROPERTY_TYPE}.npy")
    full_log_probs_filename = os.path.join(args.output_dir, f"mcmc_full_log_probs_{MODEL_PROPERTY_TYPE}.npy")
    np.save(full_chain_filename, full_chain)
    np.save(full_log_probs_filename, full_log_probs)
    logging.info(f"\nFull MCMC chain saved to '{full_chain_filename}'")
    logging.info(f"Full MCMC log-probabilities saved to '{full_log_probs_filename}'")

    logging.info("\nMCMC run complete.")

    flat_samples = sampler.get_chain(flat=True)
    log_probs_flat = sampler.get_log_prob(flat=True)

    logging.info(f"Shape of flattened chain: {flat_samples.shape}")
    logging.info(f"Shape of flattened log probabilities: {log_probs_flat.shape}")

    param_names_for_print = list(PRIOR_BOUNDS.keys())
    logging.info("\n--- MCMC Results Summary (Mean and Std Dev of parameters) ---")
    for i, name in enumerate(param_names_for_print):
        mean_val = np.mean(flat_samples[:, i])
        std_val = np.std(flat_samples[:, i])
        logging.info(f"{name}: {mean_val:.4f} +/- {std_val:.4f}")

    output_filename = os.path.join(args.output_dir, f"mcmc_chain_results_{MODEL_PROPERTY_TYPE}.npy")
    np.save(output_filename, flat_samples)
    logging.info(f"\nMCMC chain (flattened) saved to '{output_filename}'")

    output_log_probs_filename = os.path.join(args.output_dir, f"mcmc_log_probs_{MODEL_PROPERTY_TYPE}.npy")
    np.save(output_log_probs_filename, log_probs_flat)
    logging.info(f"MCMC log-probabilities (flattened) saved to '{output_log_probs_filename}'")


    # --- START OF VISUALIZATION CODE ---
    logging.info("\nGenerating corner plot...")
    corner_plot_labels = [
        r"$\log_{10}(R_{\rm all\,sky})$",
        r"$\log_{10}(E_{\min})$",
        r"$\gamma$",
        r"$\log_{10}(E_\star)$",
        r"$\alpha$"
    ]

    fig_corner = corner.corner(
        flat_samples,
        labels=corner_plot_labels,
        quantiles=[0.16, 0.5, 0.84],
        show_titles=True,
        title_fmt=".2f",
        verbose=False,
        hist_kwargs={"density": True},
        plot_datapoints=False,
        fill_contours=True,
        cmap="viridis",
    )

    corner_plot_filename = os.path.join(args.output_dir, f"corner_plot_{MODEL_PROPERTY_TYPE}.png")
    plt.savefig(corner_plot_filename, dpi=200, bbox_inches='tight')
    logging.info(f"Corner plot saved to '{corner_plot_filename}'")
    plt.close(fig_corner)

    logging.info("\n--- MCMC Results Summary (Median and 68% Credible Intervals) ---")
    for i, name in enumerate(param_names_for_print):
        mcmc_quantiles = np.percentile(flat_samples[:, i], [16, 50, 84])
        lower_err = mcmc_quantiles[1] - mcmc_quantiles[0]
        upper_err = mcmc_quantiles[2] - mcmc_quantiles[1]
        txt = f"{name}: ${mcmc_quantiles[1]:.3f}^{{+{upper_err:.3f}}}_{{-{lower_err:.3f}}}$"
        logging.info(txt)

    logging.info("\nGenerating trace plots...")
    fig_trace, axes_trace = plt.subplots(ndim + 1, figsize=(10, 2.5 * (ndim + 1)), sharex=True)

    for i in range(ndim):
        ax = axes_trace[i]
        ax.plot(full_chain[:, :, i], "k", alpha=0.3)
        ax.set_ylabel(corner_plot_labels[i])
        ax.yaxis.set_label_coords(-0.1, 0.5)

    ax_log_prob = axes_trace[ndim]
    ax_log_prob.plot(full_log_probs, "k", alpha=0.3)
    ax_log_prob.set_ylabel(r"$\ln P$")
    ax_log_prob.yaxis.set_label_coords(-0.1, 0.5)

    axes_trace[-1].set_xlabel("Steps")
    fig_trace.tight_layout()

    trace_plot_filename = os.path.join(args.output_dir, f"trace_plots_{MODEL_PROPERTY_TYPE}.png")
    plt.savefig(trace_plot_filename, dpi=200, bbox_inches='tight')
    logging.info(f"Trace plots saved to '{trace_plot_filename}'")
    plt.close(fig_trace)

    try:
        tau = sampler.get_autocorr_time()
        logging.info(f"\nAutocorrelation times (tau) per parameter: {tau}")
        effective_samples_est = len(flat_samples) / np.max(tau)
        logging.info(f"Estimated effective number of independent samples: {effective_samples_est:.0f}")
    except emcee.autocorr.AutocorrError as e:
        logging.warning(f"\nCould not estimate autocorrelation time: {e}")
        logging.warning("This might happen if the chain is too short or not well-converged.")

    logging.info("\nAll visualization and diagnostic plots have been generated.")
    logging.info(f"Check the directory '{args.output_dir}' for the output files, including '{args.log_file}'.")
    logging.info("\n--- MCMC Script Finished Successfully ---")

Entered
Parsed args: Namespace(catalog_path='/kaggle/input/datasets/gaurav7615244/dataset/LVG_table.txt', output_dir='/kaggle/working/MCMC1_RESULT', log_file='mcmc_run.log', nwalkers=200, nsteps=2000, nburnin=500, model_property_type='sfr', nthreads=1)
INFO: Logging configured. Output will be saved to '/kaggle/working/MCMC1_RESULT\mcmc_run.log'
INFO: --- Starting Bayesian FRB Rate Modeling with emcee ---
INFO: Command-line arguments: Namespace(catalog_path='/kaggle/input/datasets/gaurav7615244/dataset/LVG_table.txt', output_dir='/kaggle/working/MCMC1_RESULT', log_file='mcmc_run.log', nwalkers=200, nsteps=2000, nburnin=500, model_property_type='sfr', nthreads=1)
INFO: CHIME Observed Rate (for likelihood): 1.918e+05 bursts / yr
CRITICAL: Error: Galaxy catalog '/kaggle/input/datasets/gaurav7615244/dataset/LVG_table.txt' not found. Please ensure the file path is correct. Exiting.


SystemExit: 1